In [1]:
!pip install -q transformers peft accelerate huggingface_hub sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.1 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel
from peft import get_peft_model, LoraConfig, TaskType
from huggingface_hub import hf_hub_download

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# =============================================
# YOUR EXACT ARCHITECTURE (DO NOT CHANGE)
# =============================================
class SingleTokenMLP(nn.Module):
    def __init__(self, paragraph_dim, decoder_hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(paragraph_dim, 2048),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(2048, decoder_hidden_dim)
        )
    def forward(self, x):
        return self.net(x)

class EndToEndInverter(nn.Module):
    def __init__(self, paragraph_dim, decoder_hidden_dim, prefix_len, decoder_model):
        super().__init__()
        self.prefix_len = prefix_len
        self.decoder_hidden_dim = decoder_hidden_dim
        self.m_parallel_mlps = nn.ModuleList([
            SingleTokenMLP(paragraph_dim, decoder_hidden_dim)
            for _ in range(prefix_len)
        ])
        self.decoder = decoder_model

    def generate(self, paragraph_embs, tokenizer, max_new_tokens=128):
        device = paragraph_embs.device
        outputs = [mlp(paragraph_embs) for mlp in self.m_parallel_mlps]
        prefix_embeds = torch.stack(outputs, dim=1)
        prefix_embeds = prefix_embeds.to(device=device, dtype=self.decoder.dtype)
        output_ids = self.decoder.generate(
            inputs_embeds=prefix_embeds,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,
            repetition_penalty=1.2
        )
        return tokenizer.batch_decode(output_ids, skip_special_tokens=True)

Device: cuda


In [3]:
MODEL_NAME = "Qwen/Qwen3-0.6B"
PARAGRAPH_DIM = 1024
PREFIX_LEN = 32

# Step 1: Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Step 2: Load base Qwen decoder
base_decoder = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map={"": device}
)

# Step 3: Apply the EXACT same LoRA config used during training
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"]
)
decoder_with_lora = get_peft_model(base_decoder, lora_config)

# Step 4: Build the inverter model
model = EndToEndInverter(
    paragraph_dim=PARAGRAPH_DIM,
    decoder_hidden_dim=base_decoder.config.hidden_size,
    prefix_len=PREFIX_LEN,
    decoder_model=decoder_with_lora
)

# Step 5: Download and load the .pt weights from HuggingFace
print("Downloading trained weights from HuggingFace...")
pt_path = hf_hub_download(
    repo_id="Subhav-K/qwen-embedding-inverter-v7",       # Your repo!
    filename="entire_end_to_end_model_100k.pt"            # Your .pt file name!
)

state_dict = torch.load(pt_path, map_location="cpu")
model.load_state_dict(state_dict)

model = model.to(device)
model.eval()
print("Model loaded and ready for inference!")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

entire_end_to_end_model_100k.pt:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Model loaded and ready for inference!


In [4]:
# This is the SAME encoder used to create your training dataset embeddings
ENCODER_NAME = "Qwen/Qwen3-Embedding-0.6B"

print("Loading the encoder model...")
enc_tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME, trust_remote_code=True)
enc_model = AutoModel.from_pretrained(ENCODER_NAME, torch_dtype=torch.float32).to(device)
enc_model.eval()

def encode_paragraphs(texts):
    """Takes a list of English paragraphs and converts them into 1024-dim embeddings."""
    inputs = enc_tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = enc_model(**inputs)
    
    # Mean pooling (same logic as your training data pipeline)
    token_embeds = outputs.last_hidden_state
    mask = inputs["attention_mask"].unsqueeze(-1).float()
    sentence_embeds = (token_embeds * mask).sum(dim=1) / mask.sum(dim=1)
    sentence_embeds = F.normalize(sentence_embeds, p=2, dim=1)
    
    return sentence_embeds

print("Encoder ready!")

Loading the encoder model...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Encoder ready!


In [5]:
# =============================================
# PUT YOUR 5 TEST PARAGRAPHS HERE!
# =============================================
test_paragraphs = [
    "Paris is the capital of France and it is known for its rich cultural heritage, world-renowned art museums, historic architecture, and famous landmarks like the Eiffel Tower and the Louvre Museum.",
    
    "The Earth revolves around the Sun once every year in an elliptical orbit, and this motion along with its tilted axis is responsible for the changing seasons we experience.",
    
    "Water boils at one hundred degrees Celsius under standard atmospheric pressure, and this physical property is widely used in cooking, scientific experiments, and industrial processes.",
    
    "Mount Everest is the highest mountain above sea level, standing at over 8800 meters, and it is located in the Himalayas on the border between Nepal and China.",
    
    "The human brain controls the entire body by sending electrical and chemical signals through the nervous system, allowing us to think, move, feel emotions, and process sensory information."
]

# Step 1: Encode the paragraphs into 1024-dim embeddings
print("Encoding paragraphs into embeddings...")
with torch.no_grad():
    embeddings = encode_paragraphs(test_paragraphs)
print(f"Embedding shape: {embeddings.shape}")  # Should be [5, 1024]

# Step 2: Feed embeddings through the inverter to reconstruct text
print("\nGenerating reconstructions...\n")
print("=" * 80)

with torch.no_grad():
    generated_texts = model.generate(embeddings, tokenizer, max_new_tokens=60)

# Step 3: Display results
for i in range(len(test_paragraphs)):
    print(f"\n=== Sample {i+1} ===")
    print(f"ORIGINAL : {test_paragraphs[i][:200]}")
    print(f"GENERATED: {generated_texts[i]}")
    print("-" * 80)

print("\nDone!")

Encoding paragraphs into embeddings...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Embedding shape: torch.Size([5, 1024])

Generating reconstructions...


=== Sample 1 ===
ORIGINAL : Paris is the capital of France and it is known for its rich cultural heritage, world-renowned art museums, historic architecture, and famous landmarks like the Eiffel Tower and the Louvre Museum.
GENERATED: Paris is a world-renowned city with many attractions. Paris has the most famous landmarks, including the Eiffel Tower and Louvre Museum. The city also boasts numerous museums, art galleries, theaters, and historical sites.
--------------------------------------------------------------------------------

=== Sample 2 ===
ORIGINAL : The Earth revolves around the Sun once every year in an elliptical orbit, and this motion along with its tilted axis is responsible for the changing seasons we experience.
GENERATED: The Earth rotates around the Sun in a circular path. The motion of the Earth is called its orbit, and it takes about 365 days to complete one full revolution around the Sun.
---